# SFINCS — NJ Sandy: forcing phase-lag viewer

The modeled pre-storm tide peaks **late** vs observations, because the northern boundary is
interpolated from the harbor-phase **Battery** gauge (Sandy Hook was excluded — it failed
mid-storm). This notebook A/Bs boundary **forcing sources** to re-phase the coast, keeping the
sealed-premier build/waves fixed so only the forcing changes.

**Status 2026-07-22.** Control is the premier **`faber-waves-premier`**; every arm's `sfincs.inp` is
identical to it key-for-key, so the *only* difference is the boundary forcing file.

| arm | forcing | verdict |
|---|---|---|
| **`phaselag_composite`** (v1) | `noaa_sandy_composite` | phase **won**, level **lost** — §4. Not adopted. |
| **`phaselag_composite_v2`** | `noaa_sandy_composite_v2` | ⏳ SLURM **58882823** — the arm that isolates phase from level |

v1 won on phase (Sandy Hook +17.6 → **+7.8** min, Shrewsbury +36.9 → **+25.5**) but lost on level
(HWM bias +0.32 → **+0.73** m, within-0.5 m 74% → **21%**) because adding the Sandy Hook node also
carried the Battery's *unscaled* surge, lifting the interpolated mid-coast ~+0.2 m. v2 keeps the
local harmonic **tide** but takes the **NTR** as the Battery→AC interpolant, so the node lands on
the existing surge line and only the tide changes. **Read §4 before adopting anything.**

> ⚠️ The earlier three arms (`phaselag_battery` / `_shblend` / `_gtsm`) were **void and deleted**:
> they were staged from the pre-rebuild *leaking* grid. Do not resurrect those labels.

The estuary leak/Shark-carve story lives in the archived viewer
`archive/notebooks/sfincs-nj-sandy-viz-estuary-leakfix.ipynb`.

## Setup

In [ ]:
# Viz stack (this import also primes PROJ before hydromt loads).
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from hydromt_sfincs import SfincsModel
from nj_sfincs import plots, validate

EXP_ROOT = ROOT / "experiments"
print("experiments dir:", EXP_ROOT)

## 1. Offshore tidal phase by forcing source — the cheap headline (no SFINCS run)

Each source's series nearest the northern anchor (the Sandy Hook gauge) is cross-correlated
against the **real** Sandy Hook observed tide. **Positive = the source is phase-late at the coast**
and will import a late tide (like the Battery); a source near 0 delivers the observed offshore phase.
Sources whose data file isn't built yet show `n/a`.

Early result (built sources): Battery **+21 min**, Sandy Hook blend **0 min**.

In [ ]:
# label -> data_catalog geodataset key
SOURCES = {
    "NOAA Battery (baseline)": "noaa_sandy_nj",
    "NOAA composite v1 (arm)": "noaa_sandy_composite",
    "NOAA composite v2 (arm)": "noaa_sandy_composite_v2",
    "NOAA Sandy Hook blend": "noaa_sandy_nj_shblend",
    "GTSM-ERA5 total": "gtsm_sandy",
    "GTSM tide-only": "gtsm_sandy_tide",
    "FES2014 tide": "fes_sandy_tide",
}
plots.plot_source_phase(SOURCES);

## 2. Modeled gauge phase — Battery vs blend (vs GTSM)

Each run's label carries its pre-storm phase lag as `Δφ +NN min` (+ = model peaks later than obs).
Read the **left** of the "record ends" line (the pre-storm tide) for the interior gauges. Only runs
already on disk are shown.

In [ ]:
# label -> experiment dir under experiments/ (see nj_sfincs.config EXPERIMENTS).
# CONTROL FIRST: the premier is the baseline, not a deleted phaselag_* arm.
# Arms not yet on disk are dropped, so this stays runnable while v2 is still in the queue.
BA = {
    "premier (Battery)": "faber-waves-premier",
    "composite v1": "phaselag_composite",
    "composite v2": "phaselag_composite_v2",
}
BA = {k: v for k, v in BA.items() if (EXP_ROOT / v / "sfincs_map.nc").exists()}
print("runs present:", list(BA) or "(none — expected faber-waves-premier + phaselag_composite)")
if BA:
    plots.plot_gauge_verification(BA);

## 3. Phase-lag table (minutes; + = model late)

`validate.gauge_phase_lag` per run: Sandy Hook + Shrewsbury from the 10-min his, Shark from the
hourly map at wet channel cells.

Measured (2026-07-22):

| gauge | premier | composite | Δ |
|---|---|---|---|
| Sandy Hook | +17.6 | **+7.8** | −9.8 |
| Shrewsbury | +36.9 | **+25.5** | −11.4 |
| Shark River | +32.8 | +35.1 | +2.3 |

The coastal lag more than halves, and the Shrewsbury interior improves by *the same ~10 min* —
i.e. the interior lag was largely **imported** from the boundary, not generated by conveyance.
But Sandy Hook lands at **+7.8**, not the ~0 that `source_phase_lag` measures at the source: that
residual ~8 min is propagation across the shelf, and it is the honest ceiling of a forcing-only
fix. Shark is unmoved (its lag resolution is hourly, so +2.3 min is inside the noise).

In [ ]:
rows = {}
for label, name in BA.items():
    d = EXP_ROOT / name
    mod = SfincsModel(str(d), data_libs=[str(ROOT / "data" / "data_catalog.yml")], mode="r")
    validate.read_output(mod)  # loads his + map, no floodmap downscale
    rows[label] = validate.gauge_phase_lag(mod, d)
pd.DataFrame(rows).T if rows else print("no runs yet")

## 4. Regression — did re-phasing hurt the crest or the flood extent?

**Yes, for v1. It is not adoptable as-is.**

| | premier (control) | composite v1 | verdict |
|---|---|---|---|
| Shrewsbury gauge (obs 2.935) | 2.837 (**−0.10**) | 3.186 (**+0.25**) | overshoots |
| HWM bias | +0.318 | **+0.732** | worse |
| HWM RMSE | 0.480 | **0.813** | worse |
| HWM within 0.5 m | **74 %** | **21 %** | much worse |
| **SSS 2258 Sea Bright (obs 3.465)** | **3.650 (+0.19)** | **4.006 (+0.54)** | **worse** |
| MOTF CSI | 0.706 | 0.768 | "better" — but see below |
| MOTF POD / FAR | 0.799 / 0.141 | 0.919 / 0.177 | POD-driven |

**The CSI gain is an artifact, not skill.** POD jumps 0.80 → 0.92 while HWMs say the water is
+0.73 m too high: the model floods *more*, which a wet-heavy extent metric rewards and the
elevation metric convicts. When CSI and HWM disagree, believe HWM — CSI cannot see how deep.

Three independent observations agree in one direction — HWMs, the SSS open-coast sensor, and the
Shrewsbury surveyed crest all say **v1 over-forces the shoreline**.

> ⚠️ **Sandy Hook proves nothing here.** Its observed 2.81 m is a *floor*, not a crest — the gauge
> died ~1 h before the peak. v1's higher full peak (3.43 vs 3.15) is unfalsifiable. Only the
> pre-fail rising limb (2.496 → 2.627 vs obs 2.808) is a real comparison, and it is small.
>
> ⚠️ **Methodological trap, recorded because I fell into it.** Do **not** compare the *boundary*
> support-point interpolation to a *nearshore* gauge: the model gains ~0.6 m between the boundary
> and the shore, which makes an over-forced coast look under-forced. Sample the model **at the obs
> point** (`usgs_stormtide_sea_bright` in `sfincs_his.nc`). My first reading of this SSS did exactly
> that and concluded the opposite of the truth — that the coast wanted *more* water.

### Why v1 failed — it changed two things, not one

The recipe `total = harmonic tide + NTR` is applied at every station, but where a station uses
**its own** NTR it is algebraically an identity (`tide + (obs − tide) = obs`). Verified: v1 is
`0.000000 m` different from the premier's forcing at the Battery, Atlantic City and Cape May.

So v1's *entire* diff was **one added node at Sandy Hook**, and that node did two things at once:

- **phase** — a correct open-coast tide finally anchors the boundary beside the study area
- **level** — its 3.389 m peak, carrying the Battery's NTR **unscaled**, sits +0.243 m above the
  3.146 m the premier's two-node line already implied there, lifting the interpolated mid-coast

| basin | Δ boundary level | Δ HWM bias |
|---|---|---|
| atlantic_oceanfront | +0.214 | +0.337 |
| shrewsbury_navesink | +0.222 | +0.321 |
| sandy_hook_bay | +0.234 | +0.258 |
| south_coast | +0.196 | **+0.778** ← nonlinear overwash; also why POD jumped |

That the Battery's surge is too big for Sandy Hook is expected: the support points sit at the
*gauge coordinates*, and the Battery (−74.0142, 40.7006) is **inside New York Harbor**, where the
funnel amplifies surge. Sandy Hook is only **w = 0.164** of the way along the Battery→Atlantic City
chord, so transplanting the full Battery NTR is a large over-forcing.

> ⚠️ **"Keep the recipe but use only Battery + AC" would lose the phase fix entirely** — by the
> identity above, that returns the premier exactly. The phase win comes specifically from
> substituting **Sandy Hook's tide**.

### v2 — the fix (running: SLURM 58882823)

Split the two halves by their actual spatial behaviour, which is the real content of the
Wahl/Maduwantha decomposition:

- **tide** varies sharply in phase over short distances → take it **locally** (Sandy Hook's own
  harmonic prediction; it does not need the gauge to have survived)
- **NTR** is spatially smooth → **interpolate** it along the Battery→AC chord, exactly as the
  boundary already did

Because the inserted NTR *is* the interpolant of its neighbours, the node lies **on** the existing
surge line — and adding a point on a line does not move the line. The surge field is left as the
premier had it and **only the tide changes**. No fitted parameter (the alternative, scaling the
donor NTR by ~0.91 to hit a target peak, would be calibration wearing a diagnostic's clothes).

| pre-flight check | result |
|---|---|
| identity at own-NTR stations vs premier forcing | max abs diff **0.000000 m** |
| Sandy Hook peak vs the 3.146 m already implied there | **3.143 (−0.004 m)**; v1 was **+0.243** |
| `source_phase_lag` (+ = late vs real SH obs) | premier **+21.1**, v1 −2.6, **v2 −3.3 min** |
| interpolated NTR vs SH's own observed NTR (717 samples) | **corr 0.9974**, bias +0.088 m |
| domain / `sfincs.inp` | sealed `45f4f74ca9a2347d`; inp **identical** to control |

Built by `scripts/build_noaa_composite_v2.py` → catalog key `noaa_sandy_composite_v2`.

> ⚠️ **Cadence caveat for scoring.** The composites are 6-min, the control forcing is hourly, so the
> flanking nodes read +0.021 m (Battery) and +0.038 m (AC) higher purely because hourly sampling
> misses the true peak. This is in **both** composite arms ⇒ **v2-vs-v1 is the clean single-variable
> comparison**; v2-vs-control also carries this small cadence lift.
>
> ⚠️ **Staging trap.** `prepare_experiment` drops `crsfile = sfincs.crs` and resets `storevel 1 → 0`.
> Patch both back and `diff <(sort ctrl/sfincs.inp) <(sort new/sfincs.inp)` before submitting, and
> submit the staged dir directly — `--slurm` re-runs `prepare_experiment` and wipes the patch.

In [ ]:
if BA:
    plots.plot_hwm_residual_panels(BA);
    plots.plot_motf_panels(BA);